# Tâche 1 : Question-réponse avec un modèle génératif pré‑entraîné sans adaptation

L'objectif de cette tâche est d'observer les limites d’un modèle de langage **pré‑entraîné** lorsqu’on lui pose des questions sur des sujets **potentiellement absents** de ses données d’origine.

Dans ce *notebook*, vous faites **uniquement de l’inférence** (c.-à-d. générer des textes) avec un modèle déjà  pré-entraîné. Les paires **questions-réponses** pour évaluer le modèle vous sont fournies (voir fichier *questions.json*). Cette tâche consiste à utiliser les questions comme *prompt*, générer des réponses avec le modèle non modifié, puis évaluer la pertinence des résultats à partir des réponses de référence. Aucun entraînement de modèle n'est effectué pour cette tâche. 

> Il est recommandé de faire ce travail pratique en utilisant une carte graphique GPU compatible avec HuggingFace/Pytorch.
> Si votre machine n’en possède pas, vous pouvez utiliser **Google Colab** pour exécuter le *notebook* dans le cloud.

## 1. Chargement du modèle et du tokeniser

In [ ]:
model_name = "EleutherAI/pythia-410m"

## 2. Fonctions de test question-réponse  

## 3. Génération des réponses aux questions avec le modèle

Note : N'oubliez pas d'ajouter un fichier des réponses générées par le modèle dans votre remise du travail. 

In [ ]:
test_fn = "data/test_questions.json"

## 4. Évaluation des résultats 

Voici du code que vous pouvez utiliser pour évaluer les valeurs de précision, rappel et F1 au niveau de la correspondance des jetons entre 2 séquences de textes. Dans ce cas-ci, on évalue la correspondance entre une réponse de référence (la *bonne* réponse) et la réponse générée par le modèle. Sinon, la librairie *evaluate* de HuggingFace offre des métriques similaires (métrique *squad*).  

In [ ]:
import string
import re
from collections import Counter

def remove_articles(text):
    return re.sub(r'\b(a|an|the)\b', ' ', text)

def white_space_fix(text):
    return ' '.join(text.split())

def remove_punc(text):
    exclude = set(string.punctuation)
    return ''.join(ch for ch in text if ch not in exclude)

def lower(text):
    return text.lower()

def normalize_answer(s):
    """Mettre en minuscule et retirer la ponctuation, des déterminants and les espaces."""
    return white_space_fix(remove_articles(remove_punc(lower(s))))

In [ ]:
def evaluate_f1(ground_truth, prediction):
    """Normalise les 2 textes, trouve ce qu'il y a en commun et estime précision, rappel et F1."""
    prediction_tokens = normalize_answer(prediction).split()
    ground_truth_tokens = normalize_answer(ground_truth).split()
    common = Counter(prediction_tokens) & Counter(ground_truth_tokens)
    num_same = sum(common.values())
    if num_same == 0:
        return 0.0, 0.0, 0.0
    precision = 1.0 * num_same / len(prediction_tokens)
    recall = 1.0 * num_same / len(ground_truth_tokens)
    f1 = (2 * precision * recall) / (precision + recall)
    return precision, recall, f1